# State AI Policy: shared data preparation

Prepare a common table for the supervised and topic analyses. Normalize text and dates,
remove repeated cleaned bill text, and add sponsor features before choosing model inputs.

Source: Bhargava and Soliman, [U.S. State AI Legislation Corpus, version 4](https://zenodo.org/records/20781731), `Final_Database_10.csv`.
The repository stores the unchanged file as `data/raw/Final_Database.csv`.

## 1. Set up

Install the packages in the root `requirements.txt` once. Open this notebook from the repository root or `notebooks/`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

project_dir = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(project_dir / "src"))
from prepare import clean_text, prepare_data

source_path = project_dir / "data/raw/Final_Database.csv"
output_path = project_dir / "data/processed/common_intake/common_records.csv"
pd.set_option("display.max_colwidth", 70)

## 2. Prepare and save the table

The current script produces 2,567 rows and 45 columns from 2,571 raw rows and 29 fields.
It adds five cleaned text fields, two ISO dates, `source_row`, and eight sponsor features.
Four rows are removed by keeping the last occurrence of each identical cleaned bill text.

Text cleaning standardizes Unicode and whitespace. It keeps case, punctuation, numbers and negation.
`Sponsor_Party` is parsed into lists and serializes differently in the output CSV.
The raw file stays unchanged; original text remains alongside cleaned copies for retained rows.


In [ ]:
prepared = prepare_data(source_path, output_path)
raw = pd.read_csv(source_path, dtype=str, keep_default_na=False, skip_blank_lines=False)
raw["source_row"] = range(1, len(raw) + 1)
original_columns = raw.columns.drop("source_row")

display(pd.DataFrame({
    "Check": ["Raw records", "Prepared records", "Removed records", "Original fields", "Prepared fields"],
    "Result": [len(raw), len(prepared), len(raw) - len(prepared), len(original_columns), len(prepared.columns)],
}))


## 3. Compare original and cleaned text

Original text remains available beside its cleaned version. This example shows how line breaks and extra whitespace become a single space without rewriting the bill.

In [ ]:
example = prepared.loc[
    prepared["Jurisdiction"].eq("Alabama") & prepared["Bill_Number"].eq("H 208")
].iloc[0]

display(pd.DataFrame({
    "Version": ["Original bill text", "Cleaned bill text"],
    "First 200 characters": [example["Bill_Text"][:200], example["Bill_Text_clean"][:200]],
}))

## 4. Check text coverage

Bill text, NCSL descriptions, summaries and extracts are separate fields. A blank summary does not mean the bill text is missing. Keep these records in the common table; filtering belongs to the relevant analysis.

In [ ]:
text_fields = ["Bill_Text", "NCSL_Summary", "summary", "bill_extracts"]
blank_counts = prepared[text_fields].apply(lambda column: column.str.strip().eq("").sum())
coverage = pd.DataFrame({
    "Populated": len(prepared) - blank_counts,
    "Blank": blank_counts,
})
display(coverage)

both_blank = prepared["summary"].str.strip().eq("") & prepared["bill_extracts"].str.strip().eq("")
print(f"Records with both summary and extract blank: {both_blank.sum():,}")

In [ ]:
ax = coverage.plot.barh(
    stacked=True, color=["#00274C", "#FFCB05"], figsize=(8, 3.5)
)
ax.set_title("Text coverage across prepared records")
ax.set_xlabel("Number of records")
ax.set_ylabel("Text field")
ax.legend(title="Coverage", loc="upper left", bbox_to_anchor=(1.01, 1))
plt.tight_layout()
plt.show()

The same 484 prepared records have blank summaries and extracts. All 2,567 prepared
records have populated bill text and NCSL descriptions. These counts describe field coverage,
not text quality. The raw snapshot has 2,571 records; the four removed rows do not change these blank counts.

## 5. Inspect date differences and repeated text

A later bill version can contain information unavailable when the bill was introduced.
Review dates before choosing supervised inputs. Inspect repeated cleaned text in the raw file,
because the prepared table has already removed repeated values.

Four raw pairs have identical cleaned bill text. Their source URLs encode the same bill
identifier within each pair, but some metadata differ. The script keeps the last row by
position, not a verified latest version. The [deduplication review](../data/validation/deduplication_review.csv)
records removed-to-retained row links, source-file MD5 and differing fields.


In [ ]:
introduced = pd.to_datetime(prepared["introduction_date_iso"], format="%Y-%m-%d")
versioned = pd.to_datetime(prepared["version_date_iso"], format="%Y-%m-%d")

# Inspect raw text before the preparation script removes repeated values.
raw_text = raw["Bill_Text"].map(clean_text)
repeated_raw = raw_text.duplicated(keep=False) & raw_text.ne("")
removed_raw = ~raw["source_row"].isin(prepared["source_row"])

review_counts = pd.Series({
    "Blank prepared bill titles": prepared["Bill_Title_clean"].eq("").sum(),
    "Prepared text version before introduction": versioned.lt(introduced).sum(),
    "Prepared text version after introduction": versioned.gt(introduced).sum(),
    "Raw groups sharing cleaned bill text": raw_text[repeated_raw].nunique(),
    "Raw records in those groups": repeated_raw.sum(),
    "Raw records removed by preparation": removed_raw.sum(),
}, name="Count")
display(review_counts.to_frame())
display(raw.loc[repeated_raw, ["source_row", "Jurisdiction", "Bill_Number", "Year", "Bill_Status"]])

date_example = prepared.loc[
    prepared["Jurisdiction"].eq("Alaska") & prepared["Bill_Number"].eq("HCR 3"),
    ["source_row", "Bill_Number", "Jurisdiction", "introduction_date_iso", "version_date_iso"],
]
display(date_example)


Alaska HCR 3 was introduced on February 26, 2025, while its text-version date is April 11, 2025. The parsed dates make that gap visible. The team still needs to choose the appropriate text timing for its supervised analysis.

## 6. Use the prepared table

The output is one CSV. `source_row` is the one-based record position in this source snapshot, not a model feature or a verified unique-bill identifier. Read strings explicitly so empty cells and literal values such as `N/A` stay intact.

In [ ]:
common = pd.read_csv(output_path, dtype=str, keep_default_na=False)
common[["source_row", "Bill_Number", "Jurisdiction", "Bill_Status", "Bill_Text_clean"]].head()

Use the `_clean` fields when cleaned text is appropriate. Original text remains available
for surviving records. `Bill_Status` and `Bill_Status_Grouped` contain recorded legislative
outcomes; lowercase `status` describes publisher processing. The README lists the eight
sponsor features and their roles.

Each analysis documents its selected inputs, methods and evaluation.
Common preparation removes identical cleaned bill text and adds sponsor features; it does not
select targets, vectorize text, split data or fit models. Keep dated source observations separate
until the team agrees on an observation cutoff and outcome rule.


## 7. Check the train, validation and test groups

Notebook 02 explores training on 2019–2024 introductions, validation on 2025,
and testing on 2026. The table below accounts for every prepared record under
those windows. A record outside the windows stays in the shared data; its
final inclusion is a supervised-analysis decision.

`analysis_rows.csv` records each row's split, text coverage and the order of
its introduction and text-version dates. These fields support review; they
are not model inputs or proof that text was available at introduction.

In [8]:
row_checks = common[[
    "source_row", "Jurisdiction", "Bill_Number", "introduction_date_iso",
    "version_date_iso", "Bill_Status_Grouped",
]].copy()
assert row_checks["source_row"].is_unique

intro_dates = pd.to_datetime(row_checks["introduction_date_iso"], format="%Y-%m-%d")
version_dates = pd.to_datetime(row_checks["version_date_iso"], format="%Y-%m-%d")
intro_year = intro_dates.dt.year

row_checks["split_plan"] = "Outside current windows"
row_checks.loc[intro_year.between(2019, 2024), "split_plan"] = "Train 2019–2024"
row_checks.loc[intro_year.eq(2025), "split_plan"] = "Validation 2025"
row_checks.loc[intro_year.eq(2026), "split_plan"] = "Test 2026"

row_checks["text_date_order"] = "Missing date"
dated = intro_dates.notna() & version_dates.notna()
row_checks.loc[dated & version_dates.eq(intro_dates), "text_date_order"] = "Same date"
row_checks.loc[dated & version_dates.lt(intro_dates), "text_date_order"] = "Before introduction"
row_checks.loc[dated & version_dates.gt(intro_dates), "text_date_order"] = "After introduction"
row_checks["summary_blank"] = common["summary_clean"].str.strip().eq("")
row_checks["extract_blank"] = common["bill_extracts_clean"].str.strip().eq("")

split_order = ["Train 2019–2024", "Validation 2025", "Test 2026", "Outside current windows"]
split_counts = pd.crosstab(row_checks["split_plan"], row_checks["Bill_Status_Grouped"])
split_counts = split_counts.reindex(
    index=split_order, columns=["Failed", "Passed", "Pending"], fill_value=0,
)
split_counts["Records"] = split_counts.sum(axis=1)
split_counts["Pending (%)"] = (
    100 * split_counts["Pending"]
    / split_counts["Records"].replace(0, float("nan"))
).round(1)
assert split_counts["Records"].sum() == len(common)
display(split_counts)
display(row_checks.loc[row_checks["split_plan"].eq("Outside current windows")])

Bill_Status_Grouped,Failed,Passed,Pending,Records,Pending (%)
split_plan,,,,,
Train 2019–2024,327,96,299,722,41.4
Validation 2025,281,185,550,1016,54.1
Test 2026,156,41,631,828,76.2
Outside current windows,0,1,0,1,0.0


,source_row,Jurisdiction,Bill_Number,introduction_date_iso,version_date_iso,Bill_Status_Grouped,split_plan,text_date_order,summary_blank,extract_blank
1372,1377,California,S 36,2018-12-03,2019-10-08,Passed,Outside current windows,After introduction,True,True


The current windows cover 2,566 records and leave one 2018 introduction outside
them. Keep that choice explicit when reporting cohort sizes.

The 2026 group is 76.2% Pending. Always predicting Pending would therefore
achieve 76.2% accuracy on this cohort; model comparisons also need class-specific
results and metrics such as macro F1. Thirteen text-version dates precede
introduction and remain flagged separately from later versions.

Use `source_row` for joins. The split labels follow notebook 02.
They do not define cross-validation folds.

In [9]:
display(pd.crosstab(row_checks["split_plan"], row_checks["text_date_order"]).reindex(split_order, fill_value=0))

row_check_path = output_path.with_name("analysis_rows.csv")
row_checks.to_csv(row_check_path, index=False)
print("Saved row checks to", row_check_path.relative_to(project_dir))

text_date_order,After introduction,Before introduction,Same date
split_plan,,,
Train 2019–2024,305,0,417
Validation 2025,420,0,596
Test 2026,220,13,595
Outside current windows,1,0,0


Saved row checks to data\processed\common_intake\analysis_rows.csv
